# Case study for lane merging ![Merging Diagram](merging-scenario.jpg)

In [2]:
import numpy as np
from highway_branch_dyn import *
import math

In [ ]:
def merge_geometry_straight_road(N_lane, merge_lane, merge_s, merge_length):
    '''
    Generate a merging geometry with two straight parallel lanes.
    
    Parameters:
    N_lane: Number of lanes on the main highway
    merge_lane: Number of lanes on the ramp
    merge_s: X-coordinate where the merging area begins
    merge_length: Length of the merging area where the ramp lane merges into the main road
    
    Returns:
    merge_lane_ref_X1, merge_lane_ref_X2, merge_lane_ref_Y1, merge_lane_ref_Y2, merge_lane_ref_psi1, merge_lane_ref_psi2
    '''
    lane_width = 3.6  # Assuming each lane is 3.6 meters wide
    merge_point = 50
    merge_angle = 30
    # Define Control Area for main road (straight line)
    main_lane_length = merge_s + merge_length  # Total length of main road lane up to merge end
    main_lane_ref_X = np.linspace(0, main_lane_length, num=int(main_lane_length / 0.5))
    main_lane_ref_Y = np.zeros_like(main_lane_ref_X)  # Centered on Y = 0
    main_lane_ref_psi = np.zeros_like(main_lane_ref_X)  # Straight orientation

    ramp_lane_start_Y = N_lane * lane_width  # Parallel to the main road on the right side


    ramp_lane_ref_X = np.linspace(0, main_lane_length, num=int(main_lane_length / 0.5))
    ramp_lane_ref_Y = np.ones_like(ramp_lane_ref_X) * ramp_lane_start_Y
    ramp_lane_ref_psi = np.zeros_like(ramp_lane_ref_X)  # Straight orientation

    # For merging: ramp lane merges into main road
    merge_start_index = int(merge_s / 0.5)
    merge_end_index = int((merge_s + merge_length) / 0.5)

    # Adjust Y values to linearly bring ramp lane into main road lane
    ramp_lane_ref_Y[merge_start_index:merge_end_index] = np.linspace(
        ramp_lane_start_Y, 0, num=merge_end_index - merge_start_index
    )

    return main_lane_ref_X, ramp_lane_ref_X, main_lane_ref_Y, ramp_lane_ref_Y, main_lane_ref_psi, ramp_lane_ref_psi

In [ ]:
N_lane = 2
merge_lane=1
merge_s = 50
merge_R=300
merge_side = 0
merge_lane_ref_X1,merge_lane_ref_X2,merge_lane_ref_Y1,merge_lane_ref_Y2,merge_lane_ref_psi1,merge_lane_ref_psi2 = merge_geometry_straight_road(N_lane,merge_lane,merge_s,merge_R, merge_side)
print(merge_lane_ref_X1)
merge_lane_ref_Y = np.append(merge_lane_ref_Y1,merge_lane_ref_Y2)
merge_lane_ref_X = np.append(merge_lane_ref_X1,merge_lane_ref_X2)
merge_lane_ref_psi = np.append(merge_lane_ref_psi1,merge_lane_ref_psi2)
refY = interpolant('refY','linear',[merge_lane_ref_X],merge_lane_ref_Y)
refpsi = interpolant('refpsi','linear',[merge_lane_ref_X],merge_lane_ref_psi)
merge_ref = (refY,refpsi)
merge_ref

[ 0.6    1.094  1.588  2.082  2.576  3.07   3.564  4.058  4.552  5.046
  5.54   6.034  6.528  7.022  7.516  8.01   8.504  8.998  9.492  9.986
 10.48  10.974 11.468 11.962 12.456 12.95  13.444 13.938 14.432 14.926
 15.42  15.914 16.408 16.902 17.396 17.89  18.384 18.878 19.372 19.866
 20.36  20.854 21.348 21.842 22.336 22.83  23.324 23.818 24.312 24.806
 25.3   25.794 26.288 26.782 27.276 27.77  28.264 28.758 29.252 29.746
 30.24  30.734 31.228 31.722 32.216 32.71  33.204 33.698 34.192 34.686
 35.18  35.674 36.168 36.662 37.156 37.65  38.144 38.638 39.132 39.626
 40.12  40.614 41.108 41.602 42.096 42.59  43.084 43.578 44.072 44.566
 45.06  45.554 46.048 46.542 47.036 47.53  48.024 48.518 49.012 49.506]


(Function(refY:(x)->(f) LinearInterpolant),
 Function(refpsi:(x)->(f) LinearInterpolant))

In [ ]:
v0 = 15
class agent():
    '''
    Double integrater vehile model. State = [x, v]
    '''
    def __init__(self, state=[0, v0], V_length=4, V_width=2.4, dt=0.05):
        self.state = state
        self.dt = dt
        self.V_length = V_length
        self.V_width = V_width
        self.V_pred = []
    def step(self, u):
        dxdt = np.array([self.self.state[1]], u)
        self.state = self.state + dxdt*self.dt

In [ ]:
class merging_env():
    def __init__(self, NA, lane_width = 3.6, merge_angle=np.radians(10), merge_s=50, dt=0.05):
        self.dt = dt
        self.NA = NA # Number of agent
        self.Agent_set = []
        self.merge_angle=merge_angle
        self.merge_s = merge_s
        merge_lane_start = np.array([merge_s - merge_s*np.cos(merge_angle), lane_width + np.sin(merge_angle)*merge_s])
        merge_lane_ref_s1 = np.linspace(0, merge_s, num=int(merge_s/0.5), endpoint=False) # Straight portion
        merge_lane_ref_X = merge_lane_start[0] + merge_lane_ref_s1*np.cos(merge_angle)
        merge_lane_ref_Y = merge_lane_start[1] - merge_lane_ref_s1*np.sin(merge_angle)
        merge_lane_ref_psi = -np.ones(merge_lane_ref_s1.shape)*merge_angle
        merge_lane_end = merge_s + lane_width/np.sin(merge_angle)
        self.merge_lane_ref_X = merge_lane_ref_X
        self.merge_lane_ref_Y = merge_lane_ref_Y
        self.merge_lane_ref_psi = merge_lane_ref_psi
        self.ref_Y = interpolant('ref_Y', 'linear', [self.merge_lane_ref_X], self.merge_lane_ref_Y)
        self.ref_psi = interpolant('refY','linear',[self.merge_lane_ref_X],self.merge_lane_ref_psi)
        x0 = np.array([[24, v0], [15, v0]])
        for i in range(0, self.NA):
            self.veh_set.append(agent(x0[i],dt=self.dt))
    
    def step(self):
        u_set  = [None]*self.NA # Control input
        xx_set = [None]*self.NA # Predicted state
        u0_set = [None]*self.NA # Backup policies
        x_set  = [None]*self.NA # State

        self.xbackup = np.empty([0,(self.mpc.N+1)*4])
        for i in range(0,self.NV):
            z = self.veh_set[i].state
            if self.veh_set[i].state[0] > self.merge_s:
                self.laneID[i] = 0
            xx_set[i] = self.pred_model[self.laneID[i]].zpred_eval(z)

        idx0 = self.veh_set[0].backupidx
        n = self.pred_model[self.laneID[0]].n
        x1 = xx_set[0][:,idx0*n:(idx0+1)*n]

        for i in range(0, self.NV):
            if i != 0:
                hi = np.zeros(self.m[self.laneID[i]])
                if self.laneID[i] == 0:
                    for j in range(0, self.m[0]):
                        hi[j] = min(np.append(
                            veh_col(x1, xx_set[i][:, j * n:(j + 1) * n], [self.cons.L + 1, self.cons.W + 0.2]),
                            lane_bdry_h(xx_set[i][:, j * n:(j + 1) * n], self.LB[0], self.LB[1])
                        ))
                elif self.laneID[i] == 1:
                    for j in range(0, self.m[1]):
                        hi[j] = veh_col(x1, xx_set[i][:, j * n:(j + 1) * n], [self.cons.L + 1, self.cons.W + 0.2])
                self.veh_set[i].backupidx = np.argmax(hi)
            self.veh_set[i].backupidx = 0
            u0_set[i] = self.backupcons[self.laneID[i]][self.veh_set[i].backupidx](self.veh_set[i].state)


        for i in range(0, self.NV):
            u_set[i] = u0_set[i]
            self.veh_set[i].step(u_set[i])
            x_set[i] = self.veh_set[i].state





        


        